In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# Paths.

LINEAR_METRICS_CSV = Path("artifacts/probing/linear_probe/results/experiment_metrics.csv")
MHA_METRICS_CSV = Path("artifacts/probing/multi_head_attention_probe/results/experiment_metrics.csv")
PLOTS_DIR = Path("artifacts/probing/visualization/plots")
TABLES_DIR = Path("artifacts/probing/visualization/tables")

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load linear-probe metrics, plus MHA metrics if they exist.

metrics_parts = []
linear_df = pd.read_csv(LINEAR_METRICS_CSV)
linear_df["probe_type"] = "linear_probe"
metrics_parts.append(linear_df)

if MHA_METRICS_CSV.exists():
    mha_df = pd.read_csv(MHA_METRICS_CSV)
    mha_df["probe_type"] = "multi_head_attention_probe"
    metrics_parts.append(mha_df)

metrics_df = pd.concat(metrics_parts, ignore_index=True)
if "model_name" not in metrics_df.columns:
    metrics_df["model_name"] = np.where(metrics_df["feature"].eq("medsiglip_standalone"), "medsiglip_standalone", "base_medgemma")
metrics_df["layer"] = pd.to_numeric(metrics_df["layer"], errors="coerce")
print("metrics rows", len(metrics_df))


In [ ]:
plot_labels = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
if "model_name" not in metrics_df.columns:
    metrics_df["model_name"] = np.where(metrics_df["feature"].eq("medsiglip_standalone"), "medsiglip_standalone", "base_medgemma")
metrics_df["layer"] = pd.to_numeric(metrics_df["layer"], errors="coerce")

feature_colors = {
    "mean_image_token": "#0072B2",
    "last_image_token": "#D55E00",
    "final_prompt_token": "#009E73",
    "medsiglip_baseline": "#000000",
    "yes_no_logprob": "#CC79A7",
    "mha_pooled_image_token": "#56B4E9",
}
plot_model_names = [name for name in ["base_medgemma", "lora_image_first", "lora_text_first"] if name in set(metrics_df["model_name"])]
max_layer = int(metrics_df["layer"].dropna().max())
has_preprojector = metrics_df["feature"].eq("medgemma_pre_projector_mean_image_token").any()
preprojector_x = -8
predecoder_x = -4
yes_no_x = max_layer + 8
pre_ticks = [preprojector_x, predecoder_x] if has_preprojector else [predecoder_x]
pre_tick_labels = ["Pre-projector", "Pre-decoder"] if has_preprojector else ["Pre-decoder"]
layer_ticks = list(range(0, max_layer + 1, 5))
if max_layer not in layer_ticks:
    layer_ticks.append(max_layer)

for label in [None] + plot_labels:
    fig, axes = plt.subplots(len(plot_model_names), 2, figsize=(14, 4 * len(plot_model_names)), sharey=True, squeeze=False)
    for row_i, model_name in enumerate(plot_model_names):
        for col_i, prompt_order in enumerate(["image_first", "text_first"]):
            ax = axes[row_i, col_i]
            sub_df = metrics_df[(metrics_df["model_name"] == model_name) & (metrics_df["prompt_order"] == prompt_order)]
            baseline_df = metrics_df[metrics_df["feature"] == "medsiglip_standalone"]
            if label is not None:
                sub_df = sub_df[sub_df["label"] == label]
                baseline_df = baseline_df[baseline_df["label"] == label]

            mean_df = sub_df[sub_df["feature"] == "medgemma_layer_mean_image_token"]
            if not mean_df.empty:
                mean_df = mean_df.groupby("layer", as_index=False).agg(auroc=("auroc", "mean")) if label is None else mean_df.sort_values("layer")
                preprojector_df = sub_df[sub_df["feature"] == "medgemma_pre_projector_mean_image_token"]
                predecoder_df = sub_df[sub_df["feature"] == "medgemma_pre_decoder_mean_image_token"]
                x = mean_df["layer"].tolist()
                y = mean_df["auroc"].tolist()
                if not predecoder_df.empty:
                    x = [predecoder_x] + x
                    y = [predecoder_df["auroc"].mean() if label is None else predecoder_df["auroc"].iloc[0]] + y
                if not preprojector_df.empty:
                    x = [preprojector_x] + x
                    y = [preprojector_df["auroc"].mean() if label is None else preprojector_df["auroc"].iloc[0]] + y
                ax.plot(x, y, color=feature_colors["mean_image_token"], linewidth=2.0, label="Mean image token")

            last_df = sub_df[sub_df["feature"] == "medgemma_layer_last_image_token"]
            if not last_df.empty:
                last_df = last_df.groupby("layer", as_index=False).agg(auroc=("auroc", "mean")) if label is None else last_df.sort_values("layer")
                ax.plot(last_df["layer"], last_df["auroc"], color=feature_colors["last_image_token"], linewidth=2.0, label="Last image token")

            final_df = sub_df[sub_df["feature"] == "medgemma_layer_final_prompt_token"]
            if not final_df.empty:
                final_df = final_df.groupby("layer", as_index=False).agg(auroc=("auroc", "mean")) if label is None else final_df.sort_values("layer")
                ax.plot(final_df["layer"], final_df["auroc"], color=feature_colors["final_prompt_token"], linewidth=2.0, label="Final prompt token")

            mha_df = sub_df[sub_df["feature"] == "mha_pooled_image_token"]
            if not mha_df.empty:
                mha_df = mha_df.groupby("layer", as_index=False).agg(auroc=("auroc", "mean")) if label is None else mha_df.sort_values("layer")
                ax.plot(mha_df["layer"], mha_df["auroc"], color=feature_colors["mha_pooled_image_token"], linewidth=2.0, label="MHA pooled image token")

            if not baseline_df.empty:
                ax.axhline(baseline_df["auroc"].mean() if label is None else baseline_df["auroc"].iloc[0], color=feature_colors["medsiglip_baseline"], linestyle="--", linewidth=1.8, label="MedSigLIP baseline")

            yes_no_df = sub_df[sub_df["feature"] == "medgemma_yes_no_logprob"]
            if not yes_no_df.empty:
                ax.scatter([yes_no_x], [yes_no_df["auroc"].mean() if label is None else yes_no_df["auroc"].iloc[0]], color=feature_colors["yes_no_logprob"], marker="D", s=70, zorder=5, label="Yes/no logprob")

            ax.set_title(f"{model_name}: {prompt_order}" if label is None else f"{label}: {model_name}, {prompt_order}")
            ax.set_xlabel("Representation stage")
            ax.set_ylabel("AUROC")
            ax.set_xticks(pre_ticks + layer_ticks + [yes_no_x])
            ax.set_xticklabels(pre_tick_labels + [f"L{i}" for i in layer_ticks] + ["Yes/no"], rotation=35, ha="right")
            ax.grid(True, alpha=0.3)

    handles, names = [], []
    for ax in axes.ravel():
        for handle, name in zip(*ax.get_legend_handles_labels()):
            if name not in names:
                handles.append(handle)
                names.append(name)
    fig.legend(handles, names, loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.tight_layout()
    plot_name = "mean_auroc_over_layers.png" if label is None else re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_") + "_auroc_over_layers.png"
    plot_path = PLOTS_DIR / plot_name
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("saved", plot_path)


In [ ]:
# Export report tables as CSV and LaTeX.

if "model_name" not in metrics_df.columns:
    metrics_df["model_name"] = np.where(metrics_df["feature"].eq("medsiglip_standalone"), "medsiglip_standalone", "base_medgemma")
metrics_df["layer"] = pd.to_numeric(metrics_df["layer"], errors="coerce")

feature_names = {
    "medsiglip_standalone": "MedSigLIP baseline",
    "medgemma_pre_projector_mean_image_token": "Pre-projector image token",
    "medgemma_pre_decoder_mean_image_token": "Pre-decoder image token",
    "medgemma_layer_mean_image_token": "Mean image token",
    "medgemma_layer_last_image_token": "Last image token",
    "medgemma_layer_final_prompt_token": "Final prompt token",
    "medgemma_yes_no_logprob": "Yes/no logprob",
    "mha_pooled_image_token": "MHA pooled image token",
}

model_names = {
    "base_medgemma": "Base MedGemma",
    "lora_image_first": "LoRA image-first",
    "lora_text_first": "LoRA text-first",
    "medsiglip_standalone": "MedSigLIP",
}


def make_table(label=None):
    source_df = metrics_df if label is None else metrics_df[metrics_df["label"].eq(label)]
    rows = []

    medsiglip_df = source_df[source_df["feature"].eq("medsiglip_standalone")]
    if not medsiglip_df.empty:
        rows.append({
            "Model": "MedSigLIP",
            "Representation": "MedSigLIP baseline",
            "Prompt order": "--",
            "Best layer": "--",
            "AUROC": medsiglip_df["auroc"].mean(),
        })

    for model_name in ["base_medgemma", "lora_image_first", "lora_text_first"]:
        model_df = source_df[source_df["model_name"].eq(model_name)]
        if model_df.empty:
            continue

        for feature in ["medgemma_pre_projector_mean_image_token", "medgemma_pre_decoder_mean_image_token"]:
            sub = model_df[model_df["feature"].eq(feature)]
            if not sub.empty:
                rows.append({
                    "Model": model_names.get(model_name, model_name),
                    "Representation": feature_names[feature],
                    "Prompt order": "--",
                    "Best layer": "--",
                    "AUROC": sub["auroc"].mean(),
                })

        for feature in ["medgemma_layer_mean_image_token", "medgemma_layer_last_image_token", "medgemma_layer_final_prompt_token", "mha_pooled_image_token"]:
            for prompt_order in ["image_first", "text_first"]:
                sub = model_df[model_df["feature"].eq(feature) & model_df["prompt_order"].eq(prompt_order)]
                if not sub.empty:
                    best = (
                        sub.groupby("layer", as_index=False)
                        .agg(AUROC=("auroc", "mean"))
                        .sort_values("AUROC", ascending=False)
                        .iloc[0]
                    )
                    rows.append({
                        "Model": model_names.get(model_name, model_name),
                        "Representation": feature_names[feature],
                        "Prompt order": prompt_order.replace("_", "-"),
                        "Best layer": f"L{int(best['layer'])}",
                        "AUROC": best["AUROC"],
                    })

        for prompt_order in ["image_first", "text_first"]:
            sub = model_df[model_df["feature"].eq("medgemma_yes_no_logprob") & model_df["prompt_order"].eq(prompt_order)]
            if not sub.empty:
                rows.append({
                    "Model": model_names.get(model_name, model_name),
                    "Representation": "Yes/no logprob",
                    "Prompt order": prompt_order.replace("_", "-"),
                    "Best layer": "--",
                    "AUROC": sub["auroc"].mean(),
                })

    table = pd.DataFrame(rows)
    table["AUROC"] = table["AUROC"].round(3)
    return table


def bold_max_tex(table):
    tex_table = table.copy()
    max_value = tex_table["AUROC"].max()
    tex_table["AUROC"] = tex_table["AUROC"].map(lambda x: f"\\textbf{{{x:.3f}}}" if x == max_value else f"{x:.3f}")
    return tex_table.to_latex(index=False, escape=False)


all_tables = {"mean": make_table(None)}
for label in ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]:
    all_tables[re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")] = make_table(label)

for name, table in all_tables.items():
    csv_path = TABLES_DIR / f"representation_{name}.csv"
    tex_path = TABLES_DIR / f"representation_{name}.tex"
    table.to_csv(csv_path, index=False)
    tex_path.write_text(bold_max_tex(table), encoding="utf-8")
    print("saved", csv_path)
    print("saved", tex_path)
    display(table)